In [1]:
# Preliminary operation forcing Jupyter to reload all modules during execution
%load_ext autoreload
%autoreload 2

In [2]:
### Test new functionality in senpi to load in raw videos captured by EOSL users
import senpi
from senpi import *
from senpi.data_io import *
from senpi.sim.simulator import EventSimulator
from senpi.sim.params import make_params
from senpi import constants
from senpi.data_manip.conversions import *


import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from notebook_video_writer import VideoWriter

In [3]:
# Setup path and load video
import torchvision
from torchvision import transforms
vid_path = "C:/Users/jgreene97/Documents/Repositories/eosl-grip-ebi/event_generation_test_videos/RawVideoTest1.mp4"  # change on local machine
test = torchvision.io.read_video(vid_path, output_format="TCHW")[0]  # return videop but remove from tuple containing extraneous infos

# Set vid parameters
# N = test.size()[0]
N = 100
H = 500
W = 500



test = test[0:N, :]  # clip to small time
# set up transforms for video preprocessing
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.CenterCrop(size=(H, W))
    ## ADD DESIRED PREPROCESSING TRANSFORMS HERE
])

# In this example, convert to grayscale and squeeze channel dim - convert to torch float to mitigate downstream issues
test = torch.squeeze(transform(test)).to(dtype=torch.float)
test = (test - test.min()) / (test.max() - test.min())  # minmax normalize
print(test.size(), test.dtype)  # sanity check
print(test.min(), test.max())

torch.Size([100, 500, 500]) torch.float32
tensor(0.) tensor(1.)


In [4]:
# Write input video
with VideoWriter(fps=30) as vw:
    for i in tqdm(range(test.size()[0])):

        # # for visualization
        vw.add(test[i, :])

100%|██████████| 100/100 [00:00<00:00, 260.34it/s]


In [5]:
## Convert to photometric measurement - in practice will need to calibrate camera and relate DN to photons
params = make_params()
params["return_events"] = 1 # return events for stream-based filtering / reshaping
test = test * 0.5*params["well_cap"]  # for now, just assume peak associated to some amount our well cap

In [6]:
# Make event camera simulator class
es = EventSimulator(params=params)

# run video through sim
events, etest = es.forward(test)

In [ ]:
# Do visualizations
plot_time_surface(etest[1:10, :], plot_type='2d')

In [ ]:
# # Show output
# # write a video to the notebook to visualize result!
# with VideoWriter(filename="es_test.mp4", fps=30) as vw:
#     for i in tqdm(range(etest.size()[0])):

#         # # for visualization
#         vw.add((etest[i,:].detach().cpu().numpy()+1)/2) # this only shows POSITIVE values between 0 and 1

In [ ]:
# # Now perform downstream filtering
# ba_filter = BAFilter(time_threshold=1, height=H, width=W)
# filtered_frames = ba_filter.filter_frames_tensor(frames=etest, inplace=False)
# filtered_frames.backward(torch.ones_like(filtered_frames))

In [ ]:
# # Save events to txt for reconstruction
# senpi.save_df_to_csv(events, f='./events.txt', revert_polarity=False, delim=' ')  # True reverts back to 0, 1

In [ ]:
# Reshape and show filtered
# rec = events_tensor_batch_to_vol(events, total_batch_size=etest.size(0), height= 500, width= 500)
# print(rec.size())